# 03 - Gaussian Naive Bayes

Target: `DiagnosisType` (Normal / Tipe 2 / Gestasional).

Langkah:
1. Split data latih dan uji, lalu isi nilai kosong dengan median data latih
2. Hitung prior tiap kelas
3. Hitung mean dan variansi tiap fitur per kelas
4. Hitung posterior dengan Teorema Bayes
5. Pilih kelas dengan posterior terbesar

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import GaussianNB

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

df_gabungan = pd.read_csv(PROCESSED_DIR / "merged_diagnosis.csv")
df_model = df_gabungan.drop(columns=["SumberData"])

print(f"Dataset model: {df_model.shape}")
print("Nilai hilang per kolom (belum diisi):", df_model.isna().sum().to_dict())
print(f"random_state={RANDOM_STATE}, test_size={TEST_SIZE}")

Dataset model: (4293, 6)
Nilai hilang per kolom (belum diisi): {'Age': 0, 'Pregnancies': 0, 'BMI': 1092, 'BloodPressure': 35, 'Glucose': 518, 'DiagnosisType': 0}
random_state=42, test_size=0.2


## Split data dan training

Data dibagi 80:20 dengan stratify supaya proporsi kelas tetap sama. Median dihitung dari data
latih saja, lalu dipakai untuk mengisi nilai kosong di data latih dan data uji.

In [2]:
X = df_model.drop(columns=["DiagnosisType"])
y = df_model["DiagnosisType"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

median_latih = {kolom: float(X_train[kolom].median()) for kolom in X_train.columns}
X_train = X_train.fillna(value=median_latih)
X_test = X_test.fillna(value=median_latih)

model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Dilatih pada", X_train.shape[0], "sampel latih,", X_test.shape[0], "sampel uji")
print("Median data latih untuk imputasi:", median_latih)
print("Sisa nilai hilang (latih, uji):", int(X_train.isna().sum().sum()), int(X_test.isna().sum().sum()))
print("Kelas yang dipelajari model.classes_:", model.classes_.tolist())

Dilatih pada 3434 sampel latih, 859 sampel uji
Median data latih untuk imputasi: {'Age': 32.0, 'Pregnancies': 2.0, 'BMI': 28.4, 'BloodPressure': 80.0, 'Glucose': 154.0}
Sisa nilai hilang (latih, uji): 0 0
Kelas yang dipelajari model.classes_: ['Gestasional', 'Normal', 'Tipe 2']


## Cross validation dan cek parameter

Hasil satu kali split bisa kebetulan bagus atau jelek, jadi dicek juga dengan 5-fold
cross validation pada data latih. Pengisian median ada di dalam pipeline supaya median
dihitung ulang di tiap fold (tidak bocor ke fold validasi).

Parameter yang bisa diatur di Gaussian NB cuma `var_smoothing`, jadi dicoba beberapa nilai.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

hasil_cv = []
for vs in [1e-12, 1e-9, 1e-6, 1e-4, 1e-3, 1e-2]:
    pipa = make_pipeline(SimpleImputer(strategy="median"), GaussianNB(var_smoothing=vs))
    skor = cross_validate(pipa, X.loc[X_train.index], y_train, cv=cv, scoring=["accuracy", "f1_macro"])
    hasil_cv.append({
        "var_smoothing": f"{vs:.0e}",
        "akurasi": skor["test_accuracy"].mean(),
        "akurasi_std": skor["test_accuracy"].std(),
        "f1_makro": skor["test_f1_macro"].mean(),
        "f1_makro_std": skor["test_f1_macro"].std(),
    })

pd.DataFrame(hasil_cv).round(4)

,var_smoothing,akurasi,akurasi_std,f1_makro,f1_makro_std
0,1e-12,0.8614,0.0087,0.7664,0.0279
1,1e-09,0.8614,0.0087,0.7664,0.0279
2,1e-06,0.8614,0.0087,0.7664,0.0279
3,1e-04,0.8605,0.0075,0.7647,0.0275
4,1e-03,0.8553,0.0091,0.7511,0.0295
5,1e-02,0.8474,0.0064,0.6684,0.0163


Nilai default (`1e-9`) sudah termasuk yang terbaik; hasilnya sama untuk nilai yang lebih kecil
dan baru turun mulai `1e-3`. Jadi model tetap memakai parameter default. Akurasi cross
validation dengan standar deviasinya juga menunjukkan hasil split di atas tidak kebetulan.

## Prior tiap kelas

In [4]:
prior_model = pd.Series(model.class_prior_, index=model.classes_, name="prior_dari_model")
prior_manual = y_train.value_counts(normalize=True).sort_index()
prior_manual.name = "prior_manual"

tabel_prior = pd.concat([prior_model, prior_manual], axis=1)
tabel_prior["cocok"] = np.isclose(tabel_prior["prior_dari_model"], tabel_prior["prior_manual"])
tabel_prior

,prior_dari_model,prior_manual,cocok
Gestasional,0.319744,0.319744,True
Normal,0.617938,0.617938,True
Tipe 2,0.062318,0.062318,True


Prior Tipe 2 paling kecil (sekitar 6%) karena data Pima jauh lebih sedikit dari GDM.
Hasil `class_prior_` sama dengan hitungan manual.

## Mean dan variansi tiap fitur per kelas

In [5]:
fitur = list(X_train.columns)
distribusi = pd.DataFrame(
    {f"mean_{kelas}": model.theta_[i] for i, kelas in enumerate(model.classes_)},
    index=fitur,
)
for i, kelas in enumerate(model.classes_):
    distribusi[f"var_{kelas}"] = model.var_[i]
distribusi.round(2)

,mean_Gestasional,mean_Normal,mean_Tipe 2,var_Gestasional,var_Normal,var_Tipe 2
Age,36.09,30.52,36.84,35.62,47.11,123.14
Pregnancies,2.30,2.19,4.83,0.78,2.50,13.55
BMI,31.11,26.88,35.52,19.69,22.97,40.23
BloodPressure,89.96,75.36,75.88,116.54,85.95,144.71
Glucose,207.26,137.30,143.12,2158.13,557.34,883.68


## Hitung posterior secara manual

$$P(y \mid x_1, \dots, x_5) \propto P(y) \prod_{i=1}^{5} P(x_i \mid y)$$

Dihitung dalam bentuk log supaya tidak underflow, lalu dibandingkan dengan `predict_proba()`.

In [6]:
def log_pdf_normal(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)


def posterior_manual(sample, model):
    log_prior = np.log(model.class_prior_)
    log_likelihood = np.array([
        log_pdf_normal(sample.values, model.theta_[k], model.var_[k]).sum()
        for k in range(len(model.classes_))
    ])
    joint = log_prior + log_likelihood
    joint_stabil = joint - joint.max()
    probs = np.exp(joint_stabil)
    probs = probs / probs.sum()
    return probs


sampel = X_test.iloc[0]
label_sebenarnya = y_test.iloc[0]

posterior_manual_hasil = posterior_manual(sampel, model)
posterior_sklearn = model.predict_proba(sampel.to_frame().T)[0]

print("Sampel data uji (indeks 0):")
print(sampel.to_string())
print()
print(f"Label sebenarnya (DiagnosisType): {label_sebenarnya}")
print()
print(f"{'Kelas':<14}{'Posterior manual':<20}{'Posterior sklearn':<20}")
for kelas, p_manual, p_sklearn in zip(model.classes_, posterior_manual_hasil, posterior_sklearn):
    print(f"{kelas:<14}{p_manual:<20.6f}{p_sklearn:<20.6f}")

assert np.allclose(posterior_manual_hasil, posterior_sklearn, atol=1e-6), "Posterior manual tidak cocok dengan sklearn!"
print()
print("Posterior manual dan hasil predict_proba() sklearn identik (selisih < 1e-6).")

Sampel data uji (indeks 0):
Age               34.0
Pregnancies        2.0
BMI               27.0
BloodPressure     86.0
Glucose          171.0

Label sebenarnya (DiagnosisType): Gestasional

Kelas         Posterior manual    Posterior sklearn   
Gestasional   0.548224            0.548224            
Normal        0.447290            0.447290            
Tipe 2        0.004486            0.004486            

Posterior manual dan hasil predict_proba() sklearn identik (selisih < 1e-6).


## Cek prediksi untuk semua data uji

In [7]:
prediksi_manual = np.array([
    model.classes_[np.argmax(posterior_manual(X_test.iloc[i], model))]
    for i in range(len(X_test))
])
prediksi_sklearn = y_pred
cocok = (prediksi_manual == prediksi_sklearn).all()
print(f"Prediksi manual identik dengan model.predict() untuk seluruh {len(prediksi_sklearn)} data uji: {cocok}")

Prediksi manual identik dengan model.predict() untuk seluruh 859 data uji: True


## Simpan model

In [8]:
import json

joblib.dump(model, MODELS_DIR / "gnb_diagnosis.joblib")

with open(MODELS_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(fitur, f, indent=2)

with open(MODELS_DIR / "imputation_medians.json", "w", encoding="utf-8") as f:
    json.dump(median_latih, f, indent=2)

print("Tersimpan:")
print(" ", MODELS_DIR / "gnb_diagnosis.joblib")
print(" ", MODELS_DIR / "feature_columns.json", "->", fitur)
print(" ", MODELS_DIR / "imputation_medians.json", "->", median_latih)

Tersimpan:
  D:\diabetes-risk-gnb\models\gnb_diagnosis.joblib
  D:\diabetes-risk-gnb\models\feature_columns.json -> ['Age', 'Pregnancies', 'BMI', 'BloodPressure', 'Glucose']
  D:\diabetes-risk-gnb\models\imputation_medians.json -> {'Age': 32.0, 'Pregnancies': 2.0, 'BMI': 28.4, 'BloodPressure': 80.0, 'Glucose': 154.0}
